# Lab 1: Baseline Notebook — F1 Top 10 (2022-2024)

**Team:** feligna  
**Goal:** Build one rule-based baseline (no ML code), evaluate it honestly on the validation set, and document leakage-aware decisions.

This notebook covers required baseline requirements (4.1–4.5) and optional stretch analysis (4.6–4.8).

In [6]:
# ── Reproducibility + Imports (MUST be first cell) ──────────────────
import sys, random, time, importlib, subprocess
import numpy as np

RANDOM_SEED = 414
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

REQUIRED = {
    'pandas': 'pandas',
    'requests': 'requests',
}

missing = []
for mod, pip_name in REQUIRED.items():
    try:
        importlib.import_module(mod)
    except ModuleNotFoundError:
        missing.append(pip_name)

if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + missing)

import pandas as pd
import requests

print(f'Python: {sys.version.split()[0]}')
print(f'Seed  : {RANDOM_SEED}')
print('Environment ready ✓')

Python: 3.13.5
Seed  : 414
Environment ready ✓


## 1. Data Ingestion (API only)

We fetch race results from Jolpica API for seasons 2022, 2023, and 2024.

In [7]:
def get_season_results(year: int, max_retries: int = 3) -> pd.DataFrame:
    all_races = []
    offset = 0
    limit = 100

    while True:
        url = f"https://api.jolpi.ca/ergast/f1/{year}/results.json?limit={limit}&offset={offset}"

        for attempt in range(max_retries):
            try:
                time.sleep(1.0)
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                break
            except requests.exceptions.RequestException as e:
                wait = 2 ** attempt * 2
                if attempt < max_retries - 1:
                    time.sleep(wait)
                else:
                    raise RuntimeError(f'API failed for {year} at offset {offset}') from e

        data = response.json()['MRData']
        total = int(data['total'])
        races = data['RaceTable']['Races']
        all_races.extend(races)
        offset += limit
        if offset >= total:
            break

    rows = []
    for race in all_races:
        for result in race['Results']:
            rows.append({
                'season': int(race['season']),
                'round': int(race['round']),
                'race_date': race['date'],
                'driver_id': result['Driver']['driverId'],
                'constructor': result['Constructor']['name'],
                'grid': int(result['grid']) if result['grid'] != '0' else np.nan,
                'position': int(result['position']) if result['position'].isdigit() else np.nan,
                'status': result['status'],
            })

    df = pd.DataFrame(rows)
    df['race_date'] = pd.to_datetime(df['race_date'])
    return df

print('Loading data from Jolpica API...')
frames = []
for y in [2022, 2023, 2024]:
    df_y = get_season_results(y)
    frames.append(df_y)
    print(f'  {y}: {len(df_y)} rows')

results = pd.concat(frames, ignore_index=True)
print(f'Total rows: {len(results)}')
print(f'Date range: {results["race_date"].min().date()} to {results["race_date"].max().date()}')

Loading data from Jolpica API...
  2022: 440 rows
  2023: 440 rows
  2024: 479 rows
Total rows: 1359
Date range: 2022-03-20 to 2024-12-08


## 2. Pre-race vs Post-race Feature Audit (No Leakage)

Only pre-race information can be used for prediction.

In [8]:
results['top_10'] = (results['position'] <= 10).astype(int)

feature_audit = pd.DataFrame([
    {'feature': 'grid', 'available_pre_race': 'YES', 'use_in_baseline': 'YES'},
    {'feature': 'season', 'available_pre_race': 'YES', 'use_in_baseline': 'NO'},
    {'feature': 'round', 'available_pre_race': 'YES', 'use_in_baseline': 'NO'},
    {'feature': 'constructor', 'available_pre_race': 'YES', 'use_in_baseline': 'NO (for now)'},
    {'feature': 'position', 'available_pre_race': 'NO', 'use_in_baseline': 'NO (leakage)'},
    {'feature': 'status', 'available_pre_race': 'NO', 'use_in_baseline': 'NO (leakage)'},
    {'feature': 'top_10', 'available_pre_race': 'NO', 'use_in_baseline': 'target only'},
])

print('Feature audit:')
print(feature_audit.to_string(index=False))

baseline_df = results.copy()
baseline_df = baseline_df[baseline_df['grid'].notna()].copy()

Feature audit:
    feature available_pre_race use_in_baseline
       grid                YES             YES
     season                YES              NO
      round                YES              NO
constructor                YES    NO (for now)
   position                 NO    NO (leakage)
     status                 NO    NO (leakage)
     top_10                 NO     target only


## 3. Temporal Split (Train / Validation / Test)

Rationale: train on past seasons, validate on recent races, keep future races untouched for final evaluation.

In [9]:
train_end = pd.Timestamp('2023-12-31')
val_end = pd.Timestamp('2024-05-31')

train = baseline_df[baseline_df['race_date'] <= train_end].copy()
val = baseline_df[(baseline_df['race_date'] > train_end) & (baseline_df['race_date'] <= val_end)].copy()
test = baseline_df[baseline_df['race_date'] > val_end].copy()

print('Split sizes:')
print(f'  train: {len(train)}')
print(f'  val  : {len(val)}')
print(f'  test : {len(test)}')

print('\nLeakage checks:')
print(f"  max(train) < min(val): {train['race_date'].max() < val['race_date'].min()}")
print(f"  max(val) < min(test): {val['race_date'].max() < test['race_date'].min()}")

Split sizes:
  train: 866
  val  : 159
  test : 319

Leakage checks:
  max(train) < min(val): True
  max(val) < min(test): True


## 4.1 Domain Heuristic Baseline

Rule: **If grid <= 10, predict Top 10; else predict Not Top 10.**

This is a pure domain rule (no ML model).

### 4.2 Accuracy reported on validation set
The next code cell applies the rule to every validation row and reports accuracy clearly.

In [10]:
# Apply heuristic ONLY on validation set
val = val.copy()
val['pred_top_10'] = (val['grid'] <= 10).astype(int)

# Accuracy (required 4.2)
val_accuracy = (val['pred_top_10'] == val['top_10']).mean()
naive_top10_acc = (val['top_10'] == 1).mean()

# Useful breakdown
tp = int(((val['pred_top_10'] == 1) & (val['top_10'] == 1)).sum())
tn = int(((val['pred_top_10'] == 0) & (val['top_10'] == 0)).sum())
fp = int(((val['pred_top_10'] == 1) & (val['top_10'] == 0)).sum())
fn = int(((val['pred_top_10'] == 0) & (val['top_10'] == 1)).sum())

print('=' * 80)
print('VALIDATION RESULTS — DOMAIN HEURISTIC BASELINE')
print('=' * 80)
print(f"Validation rows: {len(val)}")
print(f"Baseline accuracy (grid<=10 rule): {val_accuracy:.3f} ({val_accuracy*100:.1f}%)")
print(f"Naive baseline (always Top 10): {naive_top10_acc:.3f} ({naive_top10_acc*100:.1f}%)")
print('\nConfusion-style counts on validation:')
print(f'  TP={tp}, TN={tn}, FP={fp}, FN={fn}')

# Keep test untouched (defined, but not evaluated)
print(f"\nTest rows defined but not evaluated: {len(test)}")

VALIDATION RESULTS — DOMAIN HEURISTIC BASELINE
Validation rows: 159
Baseline accuracy (grid<=10 rule): 0.874 (87.4%)
Naive baseline (always Top 10): 0.503 (50.3%)

Confusion-style counts on validation:
  TP=70, TN=69, FP=10, FN=10

Test rows defined but not evaluated: 319


## 4.3 Reflection on accuracy

From this run, the heuristic baseline got **87.4% accuracy** on validation (139/159 correct), while the naive baseline (always Top 10) got **50.3%**.

As students, our take is: this looks **strong as a baseline**, but accuracy alone is still not the whole story.

What this accuracy might be hiding:
- Accuracy does not show error type balance by itself, so we still need to check FP/FN behavior.
- Even with good validation performance, race context can shift over time (incidents, upgrades, weather), so performance can move.
- This rule uses only one feature (`grid`), so it ignores other pre-race context that may matter in edge cases.

Validation error profile from this run:
- TP = 70, TN = 69, FP = 10, FN = 10
- This is relatively balanced, which gives us more confidence than accuracy alone.

So, our conclusion is: **good enough for a Lab 1 lower-bound baseline**, but not enough to claim this is the final model for decision-making.

## 4.4 Baseline as lower bound

**Lower bound decision:** In this run, the domain heuristic baseline reached **87.4% validation accuracy**.

Any Lab 2 model must beat **87.4% on validation**. If it does not beat this number, then (for this setup) it is not adding practical value over the simple `grid <= 10` rule.

## 4.5 No leakage

This baseline uses only `grid` as predictive input, which is available pre-race (after qualifying).
No post-race outcome fields (`position`, `status`, `top_10`) are used as prediction features.

Temporal leakage checks in this run were both `True` (`max(train) < min(val)` and `max(val) < min(test)`).
The test set is defined (`319` rows) but not used for baseline design/tuning.

## 4.6 Additional Metrics + 4.7 Second Baseline with sklearn (Stretch)

This section is optional (rewarded). We compute Precision, Recall, F1, ROC-AUC for:
- Domain heuristic baseline (`grid <= 10`)
- A simple sklearn baseline (Logistic Regression with 1 feature: `grid`)

Both are evaluated on the validation set only.

In [11]:
# Optional stretch dependencies (safe install guard)
import importlib, subprocess, sys
for pkg in ['scikit-learn']:
    try:
        importlib.import_module('sklearn')
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_metrics(y_true, y_pred, y_score=None):
    m = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }
    if y_score is None:
        # Fallback: use predicted labels as score proxy
        y_score = y_pred
    m['roc_auc'] = roc_auc_score(y_true, y_score)
    return m

# Ensure heuristic predictions exist
if 'pred_top_10' not in val.columns:
    val['pred_top_10'] = (val['grid'] <= 10).astype(int)

# Heuristic metrics
y_val = val['top_10'].astype(int)
heur_pred = val['pred_top_10'].astype(int)
heur_score = heur_pred.astype(float)
heur_metrics = compute_metrics(y_val, heur_pred, heur_score)

# Second baseline (sklearn): Logistic Regression with 1 pre-race feature (grid)
X_train = train[['grid']].astype(float)
y_train = train['top_10'].astype(int)
X_val = val[['grid']].astype(float)

lr = LogisticRegression(random_state=RANDOM_SEED, max_iter=1000)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_val)
lr_score = lr.predict_proba(X_val)[:, 1]
lr_metrics = compute_metrics(y_val, lr_pred, lr_score)

comparison = pd.DataFrame([
    {'model': 'Heuristic (grid<=10)', **heur_metrics},
    {'model': 'LogReg 1-feature (grid)', **lr_metrics},
]).sort_values('f1', ascending=False).reset_index(drop=True)

print('=' * 80)
print('STRETCH METRICS COMPARISON ON VALIDATION (4.6, 4.7)')
print('=' * 80)
print(comparison.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

best_model = comparison.iloc[0]['model']
best_f1 = comparison.iloc[0]['f1']
print(f"\nBest by F1 on validation: {best_model} (F1={best_f1:.3f})")

STRETCH METRICS COMPARISON ON VALIDATION (4.6, 4.7)
                  model  accuracy  precision  recall    f1  roc_auc
   Heuristic (grid<=10)     0.874      0.875   0.875 0.875    0.874
LogReg 1-feature (grid)     0.874      0.875   0.875 0.875    0.928

Best by F1 on validation: Heuristic (grid<=10) (F1=0.875)


## 4.8 Metric choice justification 

Based on this run on the validation set:
- **Heuristic (grid <= 10):** accuracy 0.874, precision 0.875, recall 0.875, F1 0.875, ROC AUC 0.874
- **LogReg (1-feature grid):** accuracy 0.874, precision 0.875, recall 0.875, F1 0.875, ROC-AUC 0.928

As students, our current opinion is:
- For a practical comparison at this point, we continue to prioritize **F1** (with precision as a backup), because it balances precision and recall in a single metric.
- In this run, both models are tied in precision/recall/F1, so F1 alone does not distinguish between them.
- The ROC-AUC is higher for LogReg, suggesting better quality in score classification, but threshold-based decisions are currently similar.

So our conclusion is:
- If we are only interested in simple threshold-based predictions, both models are similar in this case.
- If we are interested in probability-based classification and future threshold adjustment, LogReg may be more useful.
- We are not yet 100% sure which metric should take precedence in implementation; for Lab 1, F1 + precision seems like a reasonable and transparent choice to us.